![dvd_image](dvd_image.jpg)

A DVD rental company needs your help! They want to figure out how many days a customer will rent a DVD for based on some features and has approached you for help. They want you to try out some regression models which will help predict the number of days a customer will rent a DVD for. The company wants a model which yeilds a MSE of 3 or less on a test set. The model you make will help the company become more efficient inventory planning.

The data they provided is in the csv file `rental_info.csv`. It has the following features:
- `"rental_date"`: The date (and time) the customer rents the DVD.
- `"return_date"`: The date (and time) the customer returns the DVD.
- `"amount"`: The amount paid by the customer for renting the DVD.
- `"amount_2"`: The square of `"amount"`.
- `"rental_rate"`: The rate at which the DVD is rented for.
- `"rental_rate_2"`: The square of `"rental_rate"`.
- `"release_year"`: The year the movie being rented was released.
- `"length"`: Lenght of the movie being rented, in minuites.
- `"length_2"`: The square of `"length"`.
- `"replacement_cost"`: The amount it will cost the company to replace the DVD.
- `"special_features"`: Any special features, for example trailers/deleted scenes that the DVD also has.
- `"NC-17"`, `"PG"`, `"PG-13"`, `"R"`: These columns are dummy variables of the rating of the movie. It takes the value 1 if the move is rated as the column name and 0 otherwise. For your convinience, the reference dummy has already been dropped.

In [9]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error as MSE
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import RandomizedSearchCV

# Import any additional modules and start coding below
df_rental = pd.read_csv('rental_info.csv')

# Convert columns to datetime if they aren't already
df_rental['return_date'] = pd.to_datetime(df_rental['return_date'])
df_rental['rental_date'] = pd.to_datetime(df_rental['rental_date'])

# Calculate the rental length in days
df_rental["rental_length_days"] = (df_rental['return_date'] -df_rental['rental_date']).dt.days

# **Create the 2 requested dummy variable columns**
df_rental["deleted_scenes"] =  np.where(df_rental["special_features"].str.contains("Deleted Scenes"), 1,0)
df_rental["behind_the_scenes"] =  np.where(df_rental["special_features"].str.contains("Behind the Scenes"), 1,0)

# Isolate the target variable (y)
y = df_rental["rental_length_days"]

# Isolate the features (X) and drop data-leaking columns
# (We drop 'rental_length_days' because it's the target, and 'return_date'/'rental_date' because they leak the answer)
X = df_rental.drop(
    columns=["rental_length_days", "return_date", "rental_date", "special_features"]
)

# Perform the train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=9
)

# Train the Lasso model
# (We choose a tiny alpha like 0.1 so it's not too harsh, and lock the seed with random_state=9)
lasso = Lasso(alpha=0.1, random_state=9)
lasso.fit(X_train,y_train)

# Extract the feature importance (coefficients)
lasso_coef = lasso.coef_

# Identify which columns are greater than 0
# This creates a list of True/False values (True if the feature is important)
important_columns_mask = lasso_coef > 0

# Subset your data to keep ONLY the winning columns
# .iloc[:, important_columns_mask] means: Keep all rows (:), but only keep columns where mask is True
X_train_best = X_train.iloc[:, important_columns_mask]
X_test_best = X_test.iloc[:, important_columns_mask]

# See your final winning features
print("The best prediction features are:", X_train_best.columns.tolist())

# Define the dictionary of hyperparameter ranges to search over
param_dist = {
    "n_estimators": np.arange(1, 101, 1),  # Number of trees in the forest (1 to 100)
    "max_depth": np.arange(1, 11, 1),  # Maximum depth of each tree (1 to 10)
}

# Initialize the base model
rf = RandomForestRegressor(random_state=9)

# Set up RandomizedSearchCV
# (Testing random combinations across 5 cross-validation folds)
rand_search = RandomizedSearchCV(
    estimator=rf, param_distributions=param_dist, cv=5, random_state=9
)

# Fit the search to your training data
# Note: Tree models often perform better on the full X_train rather than X_train_best
rand_search.fit(X_train, y_train)

# Extract the best model and its hyperparameters
best_params = rand_search.best_params_
best_model = rand_search.best_estimator_

print("Best Hyperparameters:", best_params)

# Evaluate the winning model on your test data
y_pred = best_model.predict(X_test)
best_mse = MSE(y_test, y_pred)
print(f"Tuned Random Forest MSE: {best_mse:.2f}")

The best prediction features are: ['amount', 'amount_2', 'length_2']
Best Hyperparameters: {'n_estimators': 51, 'max_depth': 10}
Tuned Random Forest MSE: 2.23
